# SHAP Analysis of `baseline_nonlocal_vertical`

This notebook applies SHAP (SHapley Additive exPlanations) to the `baseline_nonlocal_vertical` model
to examine the interpretability of a standard baseline neural network.

The `baseline_nonlocal_vertical` model takes a single-column vertical profile of three atmospheric
fields — relative humidity (RH), equivalent potential temperature (θ_e), and saturated equivalent
potential temperature (θ_e\*) — together with three surface variables (land fraction, latent heat flux,
sensible heat flux) and maps them nonlinearly to a precipitation prediction via a 5-layer MLP.

**Goal:** Show that SHAP attributions for this model exhibit large sample-to-sample variability,
and that common summaries such as mean absolute SHAP values discard sign information that is
structurally meaningful in our kernel-based models.  This motivates the use of learned integration
kernels, which are stable across samples and preserve sign.

**Note:** SHAP is not included in the default environment.  Install it before running:
```
pip install shap
```

In [ ]:
import os
import sys
import json
import warnings
import numpy as np
import torch
import xarray as xr
import proplot as pplt
import matplotlib.ticker as mticker
from math import prod

warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

# shap must be installed separately: pip install shap
import shap

pplt.rc.update({'toplabel.weight': 'normal', 'leftlabel.weight': 'normal',
                'fontsize': 14, 'figure.dpi': 100})

## Configuration

In [ ]:
with open('../scripts/configs.json', 'r', encoding='utf-8') as f:
    CONFIGS = json.load(f)

SPLITSDIR  = CONFIGS['filepaths']['splits']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
MODELSDIR  = CONFIGS['filepaths']['models']
MODELS     = CONFIGS['models']
FIELDVARS  = CONFIGS['variables']['field']   # ['rh', 'thetae', 'thetaestar']
LOCALVARS  = CONFIGS['variables']['local']   # ['lf', 'lhf', 'shf']
TARGETVAR  = CONFIGS['variables']['target']  # 'pr'
LATRANGE   = CONFIGS['domain']['latrange']
LONRANGE   = CONFIGS['domain']['lonrange']
SEEDS      = CONFIGS['training']['seeds']    # [42, 72, 102]
SPLIT      = 'test'

# Locate the baseline_nonlocal_vertical model configuration
MODELNAME   = 'baseline_nonlocal_vertical'
MODELCONFIG = next(m for m in MODELS if m['name'] == MODELNAME)
PATCHCONFIG = MODELCONFIG['patch']   # radius=0, levmode='column', timelag=0
USELOCAL    = MODELCONFIG['uselocal']

# Global domain constraints required by PatchDataLoader
MAXRADIUS  = max(m['patch']['radius']  for m in MODELS)
MAXTIMELAG = max(m['patch']['timelag'] for m in MODELS)

# Number of features: len(FIELDVARS)+1 matches evaluate.py convention
# (+1 accounts for the single validity-mask channel appended in collate())
NFIELDVARS = len(FIELDVARS) + 1   # 4
NLOCALVARS = len(LOCALVARS)        # 3

print(f'Model            : {MODELNAME}')
print(f'Patch config     : {PATCHCONFIG}')
print(f'Use local vars   : {USELOCAL}')
print(f'nfieldvars (incl. mask): {NFIELDVARS},  nlocalvars: {NLOCALVARS}')

## Data loading

We load the test split using the same `PatchDataLoader` utilities as the main pipeline.
A subset of 700 samples is drawn for SHAP: 200 for the background reference and 500 for explanation.

In [ ]:
from scripts.data.classes import PatchDataLoader

device   = 'cpu'   # SHAP runs on CPU; set to 'cuda' if a GPU is available
splitdata = PatchDataLoader.prepare(
    [SPLIT], FIELDVARS, LOCALVARS, TARGETVAR, SPLITSDIR)

result = PatchDataLoader.dataloaders(
    splitdata, PATCHCONFIG, USELOCAL,
    LATRANGE, LONRANGE,
    batchsize=512, workers=0, device=device,
    maxradius=MAXRADIUS, maxtimelag=MAXTIMELAG)

dataset    = result['datasets'][SPLIT]
loader     = result['loaders'][SPLIT]
patchshape = dataset.shape()   # (plats=1, plons=1, plevs=nlevs, ptimes=1)
nlevs      = patchshape[2]

# Pressure levels for axis labelling
with xr.open_dataset(f'{SPLITSDIR}/{SPLIT}.h5', engine='h5netcdf') as ds:
    lev = ds['lev'].values.astype(np.float32)   # shape (nlevs,)

print(f'Patch shape        : {patchshape}')
print(f'nlevs              : {nlevs}')
print(f'Pressure levels    : {lev}')
print(f'Total test samples : {len(dataset):,}')

In [ ]:
# Collect a contiguous chunk of samples from the loader, then shuffle and split
N_BACKGROUND = 200
N_TEST       = 500
N_TOTAL      = N_BACKGROUND + N_TEST

chunks = []
for batch in loader:
    fp = batch['fieldpatch']   # (nbatch, NFIELDVARS, 1, 1, nlevs, 1)
    lv = batch['localvalues']  # (nbatch, NLOCALVARS)
    # Concatenate flattened patch + local variables into a single 2-D tensor
    X  = torch.cat([fp.flatten(1), lv], dim=1).detach()
    chunks.append(X)
    if sum(c.shape[0] for c in chunks) >= N_TOTAL:
        break

X_pool = torch.cat(chunks, dim=0)[:N_TOTAL]

rng = np.random.default_rng(42)
perm = rng.permutation(N_TOTAL)
X_bg   = X_pool[perm[:N_BACKGROUND]].float()
X_test = X_pool[perm[N_BACKGROUND:]].float()

# Feature layout in the flat vector:
#   indices [0,              nlevs)  -> RH at each pressure level
#   indices [nlevs,       2*nlevs)  -> theta_e at each pressure level
#   indices [2*nlevs,     3*nlevs)  -> theta_e* at each pressure level
#   indices [3*nlevs,     4*nlevs)  -> validity mask at each pressure level
#   indices [4*nlevs, 4*nlevs + 3)  -> lf, lhf, shf  (local surface variables)

print(f'Background tensor : {X_bg.shape}')
print(f'Test tensor       : {X_test.shape}')
print(f'Features per sample: {X_bg.shape[1]}  (= {NFIELDVARS}×{nlevs} + {NLOCALVARS})')

## Model loading

In [ ]:
from scripts.models.classes import ModelFactory

SEED = SEEDS[0]   # use the first seed; SHAP variability conclusions hold across seeds

model = ModelFactory.build(MODELNAME, MODELCONFIG, patchshape, NFIELDVARS, NLOCALVARS)
checkpoint_path = os.path.join(MODELSDIR, f'{MODELNAME}_{SEED}.pth')
state = torch.load(checkpoint_path, map_location='cpu')
model.load_state_dict(state)
model.eval()

print(f'Loaded  : {checkpoint_path}')
print(f'Params  : {sum(p.numel() for p in model.parameters()):,}')

## SHAP wrapper

`shap.DeepExplainer` requires a model that maps a single 2-D input tensor to predictions.
We wrap `BaselineNN` to accept the concatenated flat vector and reshape it internally.

In [ ]:
class FlatBaselineNN(torch.nn.Module):
    """
    Thin wrapper around BaselineNN that accepts a concatenated flat input
    [fieldpatch.flatten(1) | localvalues] and returns scalar predictions.

    The split point between fieldpatch and localvalues is fp_size = NFIELDVARS * prod(patchshape).
    """
    def __init__(self, model, nfieldvars, patchshape, nlocalvars):
        super().__init__()
        self.model      = model
        self.fp_shape   = (nfieldvars, *patchshape)   # (4, 1, 1, nlevs, 1)
        self.fp_size    = prod(self.fp_shape)          # 4 * nlevs
        self.nlocalvars = nlocalvars

    def forward(self, X):
        fp = X[:, :self.fp_size].reshape(-1, *self.fp_shape)
        lv = X[:, self.fp_size:]
        return self.model(fp, lv)   # (nbatch,)


wrapper = FlatBaselineNN(model, NFIELDVARS, patchshape, NLOCALVARS)
wrapper.eval()

# Quick sanity check
with torch.no_grad():
    y_check = wrapper(X_test[:4])
print(f'Wrapper output shape for 4 samples: {y_check.shape}')  # should be (4,)

## SHAP value computation

We use `shap.DeepExplainer`, which applies the DeepLIFT algorithm combined with SHAP's
Shapley sampling to propagate attributions back through the network layers.
The background dataset provides the reference point; SHAP values represent the contribution
of each feature *relative to the background mean*.

As an alternative, `shap.GradientExplainer` (integrated-gradients-based) can be substituted
by replacing the two lines below with:
```python
explainer  = shap.GradientExplainer(wrapper, X_bg)
shap_raw   = explainer.shap_values(X_test)
```

In [ ]:
explainer = shap.DeepExplainer(wrapper, X_bg)
shap_raw  = explainer.shap_values(X_test)   # (N_TEST, nfeatures) or list thereof

# Normalise to a plain 2-D array regardless of SHAP output format
if isinstance(shap_raw, list):
    shap_raw = shap_raw[0]
if shap_raw.ndim == 3:
    shap_raw = shap_raw[:, :, 0]
shap_raw = np.array(shap_raw, dtype=np.float32)   # (N_TEST, 4*nlevs + 3)

# Extract per-variable profiles (first 3*nlevs columns correspond to the three field variables)
# Layout: [ RH(lev0..levN) | theta_e(lev0..levN) | theta_estar(lev0..levN) | mask(lev0..levN) ]
shap_fields = shap_raw[:, :3 * nlevs].reshape(N_TEST, 3, nlevs)   # (N_TEST, 3, nlevs)
shap_local  = shap_raw[:, 4 * nlevs:]                              # (N_TEST, 3)  [lf, lhf, shf]

print(f'SHAP array shape     : {shap_raw.shape}')
print(f'Field SHAP shape     : {shap_fields.shape}  [samples, vars, levels]')
print(f'Local SHAP shape     : {shap_local.shape}   [samples, local_vars]')

## Plot 1 — Mean SHAP values and sample-to-sample variability

Each panel shows the mean SHAP attribution across 500 test samples (solid line)
together with a ±1 σ shading and a selection of 20 individual sample profiles (thin lines).
The wide spread illustrates that the network does not attribute contributions to vertical levels
in a consistent way: the same level may receive a large positive attribution for one sample
and a large negative one for another.

In [ ]:
FIELDLABELS  = ['RH', r'$\mathit{\theta_e}$', r'$\mathit{\theta_e^*}$']
N_INDIVIDUAL = 20

shap_mean = shap_fields.mean(axis=0)   # (3, nlevs)
shap_std  = shap_fields.std(axis=0)    # (3, nlevs)

rng_plot   = np.random.default_rng(0)
sample_idx = rng_plot.choice(N_TEST, N_INDIVIDUAL, replace=False)

fig, axs = pplt.subplots(
    nrows=1, ncols=3, sharey=True,
    width=8, height=5, wspace=1)

axs.format(
    collabels=FIELDLABELS,
    ylabel='Pressure (hPa)',
    xlabel='SHAP value',
    ylim=(500, 1000), yticks=100,
    yminorticks='none', grid=False,
    xticklabelsize=11, yticklabelsize=11)

for col in range(3):
    ax   = axs[col]
    mean = shap_mean[col]   # (nlevs,)
    std  = shap_std[col]    # (nlevs,)

    # Individual sample profiles — make variability immediately visible
    for i in sample_idx:
        ax.plot(shap_fields[i, col], lev, color='blue6',
                alpha=0.12, linewidth=0.7, zorder=1)

    # ±1 σ shaded envelope
    ax.fill_betweenx(lev, mean - std, mean + std,
                     color='blue6', alpha=0.30, label=r'$\pm 1\sigma$', zorder=2)

    # Mean profile
    ax.plot(mean, lev, color='blue6', linewidth=1.8,
            label='Mean', zorder=3)

    ax.axvline(0, color='k', linewidth=0.5, zorder=0)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=4, prune='both'))
    ax.xaxis.set_minor_locator(mticker.NullLocator())

axs.format(yreverse=True)
axs[0].legend(loc='ur', ncols=1, frame=False, prop={'size': 11})
pplt.show()
fig.save('../figs/shap_mean_variability.png', dpi=150)

## Plot 2 — Mean |SHAP| versus learned kernel weights

A natural summary statistic for feature importance is the mean absolute SHAP value,
which averages out sign.  The top row shows this quantity for each atmospheric variable;
the bottom row shows the normalized kernel weights from the equivalent `kernel_nonparametric_vertical`
model (same spatial and temporal context, same field variables, but with explicit learned integration).

Two differences are highlighted:
1. **Sign is lost**: mean |SHAP| is non-negative by construction; kernel weights retain sign and
   directly encode whether a given level contributes positively or negatively to precipitation.
2. **Variability is compressed**: the ±1 σ spread across seeds in the kernel weights is narrow
   (the kernel is learned stably), whereas the ±1 σ spread across test samples in |SHAP| is wide.

In [ ]:
# Load nonparametric kernel weights for the vertical-only kernel model
kernel_name = 'kernel_nonparametric_vertical'
kernel_path = os.path.join(WEIGHTSDIR, f'{kernel_name}_{SPLIT}_weights.nc')

with xr.open_dataset(kernel_path, engine='h5netcdf') as ds:
    k     = ds['k'].load()    # dims: (field, lev, seed)
    kmean = k.mean('seed')    # (field, lev)
    kstd  = k.std('seed')     # (field, lev)
    klev  = ds['lev'].values.astype(np.float32)

print(f'Kernel weights shape : {k.shape}  (fields × levels × seeds)')

In [ ]:
shap_absmean = np.abs(shap_fields).mean(axis=0)   # (3, nlevs)
shap_absstd  = np.abs(shap_fields).std(axis=0)    # (3, nlevs)

fig, axs = pplt.subplots(
    nrows=2, ncols=3,
    sharey=True, sharex='limits',
    width=8, height=7,
    wspace=1, hspace=1.5)

axs.format(
    rowlabels=['Mean |SHAP|', 'Kernel weights'],
    collabels=FIELDLABELS,
    ylabel='Pressure (hPa)',
    ylim=(500, 1000), yticks=100,
    yminorticks='none', grid=False,
    xticklabelsize=11, yticklabelsize=11)

for col in range(3):
    # ── Top row: mean |SHAP| ──────────────────────────────────────────────────
    ax   = axs[0, col]
    mean = shap_absmean[col]
    std  = shap_absstd[col]

    ax.fill_betweenx(lev, mean - std, mean + std,
                     color='blue6', alpha=0.25, label=r'$\pm 1\sigma$')
    ax.plot(mean, lev, color='blue6', linewidth=1.5, label='Mean')
    ax.set_xlabel('Mean |SHAP value|', fontsize=11)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=3, prune='both'))
    ax.xaxis.set_minor_locator(mticker.NullLocator())

    # ── Bottom row: kernel weights ────────────────────────────────────────────
    ax2       = axs[1, col]
    km_col    = kmean.isel(field=col).values
    ks_col    = kstd.isel(field=col).values

    ax2.axvline(0, color='k', linewidth=0.5)
    ax2.fill_betweenx(klev, km_col - ks_col, km_col + ks_col,
                      color='yellow6', alpha=0.25, label=r'$\pm 1\sigma$')
    ax2.plot(km_col, klev, color='yellow6', linewidth=1.5, label='Mean')
    ax2.set_xlabel('Normalized kernel weight', fontsize=11)
    ax2.xaxis.set_major_locator(mticker.MaxNLocator(nbins=3, prune='both'))
    ax2.xaxis.set_minor_locator(mticker.NullLocator())

axs.format(yreverse=True)
axs[0, 0].legend(loc='ur', ncols=1, frame=False, prop={'size': 11})
axs[1, 0].legend(loc='ur', ncols=1, frame=False, prop={'size': 11})
pplt.show()
fig.save('../figs/shap_vs_kernels.png', dpi=150)

## SHAP values for local surface variables

For completeness, the cell below shows the distribution of SHAP attributions for the three
local surface inputs (land fraction, latent heat flux, sensible heat flux).
These are scalar inputs (not vertical profiles), so variability is displayed as violin plots.

In [ ]:
LOCALLABELS = ['Land\nfraction\n(lf)', 'Latent\nheat flux\n(lhf)', 'Sensible\nheat flux\n(shf)']

fig, ax = pplt.subplots(width=5, height=3.5)
ax.format(
    ylabel='SHAP value', grid=False,
    xticklabels=LOCALLABELS, xticklabelsize=11)

positions = np.arange(NLOCALVARS)
parts = ax.violinplot(
    [shap_local[:, i] for i in range(NLOCALVARS)],
    positions=positions,
    widths=0.5, showmedians=True)

for pc in parts['bodies']:
    pc.set_facecolor('blue6')
    pc.set_alpha(0.5)
for part in ('cbars', 'cmins', 'cmaxes', 'cmedians'):
    parts[part].set_color('blue6')

ax.axhline(0, color='k', linewidth=0.5)
ax.set_xticks(positions)
pplt.show()
fig.save('../figs/shap_local_vars.png', dpi=150)

## Summary

The plots above illustrate three key limitations of SHAP-based interpretability for this baseline model:

1. **High sample-to-sample variability** (Plot 1): Individual SHAP profiles scatter widely around
   the mean, with standard deviations of comparable or larger magnitude than the mean itself.
   This reflects the fact that the MLP mixes vertical levels and variables through successive
   nonlinear layers; the attribution assigned to a given level depends heavily on the values
   of all other inputs in a given sample.

2. **Loss of sign information** (Plot 2, top row): Summarising attributions by their absolute
   values — the standard feature-importance view — discards the sign of contributions,
   making it impossible to say whether a level's anomaly *increases* or *decreases* the
   predicted precipitation for a typical sample.

3. **Contrast with kernel weights** (Plot 2, bottom row): The learned nonparametric kernel
   weights are stable across the three training seeds, preserve sign, and have a direct
   mathematical relationship to the prediction: the kernel-integrated feature *is* the
   scalar quantity passed to the nonlinear mapping.  This makes the kernel a principled,
   stable, and interpretable summary of vertical structure that SHAP attributions
   on the baseline model cannot provide.